# 🤖 Notebook 09 — Transformer Model for Reaction Prediction

**Why Transformers for chemistry?**
SMILES strings are sequences — Transformers process sequences using self-attention,
learning which atoms influence each other during reactions.

**Task:** Predict reaction outcome SMILES from reactant SMILES
(simplified seq2seq reaction prediction)

**Architecture:** SMILES Tokenizer → Transformer Encoder → MLP → Reaction class prediction

**Patent context:** Reaction prediction helps identify novel synthetic routes
that may fall outside existing process patent claims.

## STEP 1 — Install dependencies

In [ ]:
!pip install rdkit pandas numpy matplotlib scikit-learn torch --quiet
print('Done!')

## STEP 2 — SMILES tokenizer

In [ ]:
import re
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

class SMILESTokenizer:
    """
    Character-level SMILES tokenizer.
    Converts SMILES strings to integer token sequences.
    Used as input to transformer encoder.
    """
    def __init__(self):
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
        self.chars = [
            '<PAD>', '<UNK>',
            'C', 'c', 'N', 'n', 'O', 'o', 'S', 's',
            'F', 'Cl', 'Br', 'I', 'P', 'p',
            '(', ')', '[', ']', '=', '#', '-', '+',
            '1','2','3','4','5','6','7','8','9','0',
            '/', '\\', '@', '.', '%'
        ]
        self.char2idx = {c: i for i, c in enumerate(self.chars)}
        self.idx2char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)

    def tokenize(self, smiles, max_len=128):
        """Convert SMILES to padded token sequence."""
        tokens = []
        i = 0
        while i < len(smiles):
            if i + 1 < len(smiles) and smiles[i:i+2] in self.char2idx:
                tokens.append(self.char2idx[smiles[i:i+2]])
                i += 2
            elif smiles[i] in self.char2idx:
                tokens.append(self.char2idx[smiles[i]])
                i += 1
            else:
                tokens.append(self.char2idx['<UNK>'])
                i += 1
        tokens = tokens[:max_len]
        tokens += [self.char2idx['<PAD>']] * (max_len - len(tokens))
        return tokens

tokenizer = SMILESTokenizer()
print(f'Vocabulary size: {tokenizer.vocab_size}')

# Test tokenizer
test_smiles = 'CCc1ccc(Nc2ncnc3ccccc23)cc1'
tokens = tokenizer.tokenize(test_smiles)
print(f'Test SMILES: {test_smiles}')
print(f'Token length: {len(tokens)}')
print(f'First 10 tokens: {tokens[:10]}')

## STEP 3 — Load EGFR data and prepare sequences

In [ ]:
import requests
from rdkit import Chem

def fetch_egfr_data(limit=1000):
    url = 'https://www.ebi.ac.uk/chembl/api/data/activity.json'
    params = {'target_chembl_id': 'CHEMBL203', 'standard_type': 'IC50', 'limit': limit, 'offset': 0}
    records = requests.get(url, params=params).json()['activities']
    return pd.DataFrame(records)

df_raw = fetch_egfr_data(1000)
df = df_raw[['canonical_smiles', 'standard_value']].dropna()
df = df[df['standard_value'] > 0].copy()
df['pIC50'] = -np.log10(df['standard_value'].astype(float) * 1e-9)
df = df[(df['pIC50'] >= 3) & (df['pIC50'] <= 12)]

# Validate SMILES
valid_mask = df['canonical_smiles'].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)
df = df[valid_mask].reset_index(drop=True)

# Tokenize
MAX_LEN = 128
X = np.array([tokenizer.tokenize(str(s), MAX_LEN) for s in df['canonical_smiles']])
y = df['pIC50'].values.astype(np.float32)

print(f'Dataset: {len(df)} compounds')
print(f'Input shape: {X.shape}')
print(f'pIC50 range: {y.min():.2f} — {y.max():.2f}')

## STEP 4 — Build Transformer model

In [ ]:
class SMILESTransformer(nn.Module):
    """
    Transformer encoder for SMILES-based property prediction.

    Architecture:
    - Token embedding + positional encoding
    - 4-head multi-head self-attention (2 layers)
    - Mean pooling over sequence
    - MLP regression head → pIC50

    Self-attention captures long-range atom interactions
    that convolutional/fingerprint methods miss.
    """
    def __init__(self, vocab_size, embed_dim=64, nhead=4,
                 num_layers=2, max_len=128, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=128, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc1 = nn.Linear(embed_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Compute padding mask from token ids BEFORE embedding (correct API)
        padding_mask = (x == 0)                                    # [B, T] bool
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoding(positions)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        x = x.mean(dim=1)                                          # mean pooling
        x = torch.relu(self.fc1(self.dropout(x)))
        return self.fc2(x).squeeze(-1)                             # safe squeeze

model = SMILESTransformer(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64, nhead=4, num_layers=2, max_len=MAX_LEN
)
print(f'SMILESTransformer | Params: {sum(p.numel() for p in model.parameters()):,}')

## STEP 5 — Train and evaluate

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import os

os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_t = torch.LongTensor(X_train)
X_test_t  = torch.LongTensor(X_test)
y_train_t = torch.FloatTensor(y_train)
y_test_t  = torch.FloatTensor(y_test)

train_dl = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_dl  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32, shuffle=False)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = SMILESTransformer(tokenizer.vocab_size, embed_dim=64, nhead=4, num_layers=2, max_len=MAX_LEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
criterion = nn.MSELoss()

train_losses = []
for epoch in range(60):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg = epoch_loss / len(train_dl)
    train_losses.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/60 | Loss: {avg:.4f}')

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        y_true.extend(yb.numpy())
        y_pred.extend(model(xb).cpu().reshape(-1).numpy())

r2   = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f'\nTransformer → R²: {r2:.3f} | RMSE: {rmse:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, color='#1D9E75')
axes[0].set_title('Transformer Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[1].scatter(y_true, y_pred, alpha=0.5, color='#1D9E75', s=20)
axes[1].plot([3, 12], [3, 12], 'r--', alpha=0.7)
axes[1].set_xlabel('True pIC50'); axes[1].set_ylabel('Predicted pIC50')
axes[1].set_title(f'Transformer Predictions (R²={r2:.3f})', fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/transformer_training_results.png', dpi=150)
plt.show()

torch.save(model.state_dict(), 'results/metrics/transformer_model.pt')
print('All results saved!')